# 苜蓿根瘤空间转录组：单样本局部复现

本 Notebook 复现 Shao et al., *Cell* (2026) 苜蓿根瘤共转录组论文中的四项核心分析：

1. 质量检查与标准化；
2. 植物基因的独立聚类、UMAP 和空间定位；
3. Mt5.0 标记基因映射、点图和空间表达；
4. 植物与根瘤菌计数拆分及根瘤菌计数的空间占比。

分析样本是公开的 P3 切片 `FP200000569BR_C5_bottom4_MXP3.h5ad`。这是论文 20 个切片中的
一个代表切片，不是完整图谱重算。原始 FASTQ、比对、文章中的遗传学和湿实验不在范围内。

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import Markdown, display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
os.chdir(project_root)
sys.path.insert(0, str(project_root))

from analysis.reproduction import (
    Paths,
    add_qc_metrics,
    configure_logging,
    configure_publication_style,
    load_and_validate,
    load_config,
    load_marker_mapping,
    load_sample_reference_metadata,
    load_supplementary_marker_evidence,
    plot_annotation_and_depth,
    plot_cluster_confusion,
    plot_clustering,
    plot_marker_genes,
    plot_normalization_contrast,
    plot_qc_overview,
    plot_rhizobia_analysis,
    prepare_plant_analysis,
    rank_region_markers,
    resolve_inputs,
    resolve_markers,
    resolve_supplementary,
    transfer_analysis_results,
    write_run_metadata,
)

configure_logging()
configure_publication_style()
paths = Paths()
config = load_config()
sample_path, annotation_path = resolve_inputs(config)
supplementary_path = resolve_supplementary(config)
sample_reference = load_sample_reference_metadata(
    supplementary_path,
    config["data"]["sample_filename"].removesuffix(".h5ad"),
)
sample_reference.to_csv(
    paths.table_dir / "sample_reference.csv",
    index=False,
    encoding="utf-8-sig",
)

display(
    Markdown(
        f"""
**输入文件**

- 空间矩阵：`{sample_path.name}`
- Mt5.0 基因注释：`{annotation_path.name}`
- 论文补充表：`{supplementary_path.name}`

三个文件均按 SHA-256 校验，运行前会拒绝被替换或不完整的数据。
"""
    )
)
display(Markdown("作者补充表 S1-3 报告的同一样本信息："))
display(sample_reference)


def show_pdf_figure(path: Path) -> None:
    relative = path.resolve().relative_to(Path.cwd().resolve())
    display(Markdown(f"[打开矢量 PDF 图：{path.stem}]({relative.as_posix()})"))


**输入文件**

- 空间矩阵：`FP200000569BR_C5_bottom4_MXP3.h5ad`
- Mt5.0 基因注释：`Mt_gene_Acronyms_IDs_latest.xlsx`
- 论文补充表：`cell2026_mmc2.xlsx`

三个文件均按 SHA-256 校验，运行前会拒绝被替换或不完整的数据。


作者补充表 S1-3 报告的同一样本信息：

,Sample/lasso ID,Cell Number,Mean UMI,Median UMI,Mean Gene,Median Gene
5,FP200000569BR_C5_bottom4_MXP3,1270,2275.199213,1739,882.450394,751


## 1. 输入契约与质量检查

原始计数层被保留。植物和根瘤菌基因按官方 ID 前缀区分：
`MtrunA17...` 为植物，`SM-...` 为 *Sinorhizobium meliloti*。

In [2]:
adata = load_and_validate(sample_path)
adata = add_qc_metrics(adata)
display(adata)
display(adata.obs["organ"].value_counts().rename("bins").to_frame())

integrity = {
    "空间 bin": adata.n_obs,
    "基因数": adata.n_vars,
    "植物基因": int((adata.var["species"] == "Medicago").sum()),
    "根瘤菌基因": int((adata.var["species"] == "Sinorhizobium").sum()),
    "非零计数": int(adata.X.nnz),
    "总计数": int(adata.X.sum()),
}
display(Markdown("\n".join(f"- **{key}**：{value:,}" for key, value in integrity.items())))

plot_qc_overview(adata, paths.figure_dir / "01_qc_overview.pdf")
show_pdf_figure(paths.figure_dir / "01_qc_overview.pdf")

AnnData object with n_obs × n_vars = 1284 × 40160
    obs: 'nCount_Spatial', 'nFeature_Spatial', 'seurat_clusters', 'group', 'phase', 'organ', 'array_row', 'array_col', 'total_counts_recomputed', 'detected_genes_recomputed', 'plant_counts', 'rhizobia_counts', 'rhizobia_fraction', 'log10_total_counts_recomputed'
    var: 'species'
    obsm: 'spatial', 'umap'
    layers: 'X', 'counts', None (.X)

,bins
organ,
Vascular bundle,370
Cortex_2,368
Distal infection zone,158
Cortex_1,144
Proximal infection zone,81
Nitrogen fixation zone,63
Interzone,59
Meristematic zone,41


- **空间 bin**：1,284
- **基因数**：40,160
- **植物基因**：34,349
- **根瘤菌基因**：5,811
- **非零计数**：1,179,814
- **总计数**：3,088,044

[打开矢量 PDF 图：01_qc_overview](results/figures/01_qc_overview.pdf)

In [3]:
plot_annotation_and_depth(
    adata,
    paths.figure_dir / "02_annotation_and_depth.pdf",
)
show_pdf_figure(paths.figure_dir / "02_annotation_and_depth.pdf")

[打开矢量 PDF 图：02_annotation_and_depth](results/figures/02_annotation_and_depth.pdf)

## 2. 独立聚类与空间定位

聚类只使用植物基因，避免根瘤菌比例直接驱动宿主组织分组。步骤为：
过滤低表达基因、10,000 counts/ bin 标准化、`log1p`、2,000 个高变基因、20 个主成分、
15 个近邻、Leiden 聚类。随机种子固定为 42。

In [4]:
plant, plant_mask = prepare_plant_analysis(adata, config)
adata = transfer_analysis_results(adata, plant)

normalization_gene = config["analysis"]["normalization_example"]
plot_normalization_contrast(
    plant,
    normalization_gene["gene_id"],
    normalization_gene["symbol"],
    paths.figure_dir / "01b_normalization_contrast.pdf",
)
show_pdf_figure(paths.figure_dir / "01b_normalization_contrast.pdf")

metrics = plot_clustering(
    plant,
    config,
    paths.figure_dir / "03_independent_clustering.pdf",
    paths.table_dir / "clustering_metrics.json",
)
show_pdf_figure(paths.figure_dir / "03_independent_clustering.pdf")
display(Markdown("```json\n" + str(metrics) + "\n```"))

plot_cluster_confusion(
    plant,
    config,
    paths.figure_dir / "03b_cluster_annotation_confusion.pdf",
)
show_pdf_figure(paths.figure_dir / "03b_cluster_annotation_confusion.pdf")
display(
    Markdown(
        "除 ARI/NMI 外，本流程还计算 PCA 空间中的 silhouette 分数和空间 6-近邻"
        "同类比例，用于同时评估表达可分性和空间连续性。"
    )
)

discovery = rank_region_markers(
    plant,
    load_marker_mapping(annotation_path),
    paths.table_dir / "region_marker_discovery.csv",
)
display(discovery.groupby("author_region", observed=True).head(5))

[打开矢量 PDF 图：01b_normalization_contrast](results/figures/01b_normalization_contrast.pdf)

... storing 'author_tissue' as categorical


... storing 'author_cluster' as categorical


... storing 'species' as categorical


[打开矢量 PDF 图：03_independent_clustering](results/figures/03_independent_clustering.pdf)

```json
{'leiden_coarse_ari_vs_author_tissue': 0.0572143974388406, 'leiden_coarse_nmi_vs_author_tissue': 0.18951777312433554, 'leiden_coarse_n_clusters': 3.0, 'leiden_coarse_silhouette_pca20': 0.06859162449836731, 'leiden_coarse_spatial_neighbor_purity': 0.7841381100726895, 'leiden_fine_ari_vs_author_tissue': 0.1985457887267897, 'leiden_fine_nmi_vs_author_tissue': 0.2996041313547133, 'leiden_fine_n_clusters': 6.0, 'leiden_fine_silhouette_pca20': 0.06828755140304565, 'leiden_fine_spatial_neighbor_purity': 0.5786604361370716, 'author_tissue_spatial_neighbor_purity': 0.5428348909657321, 'plant_bins_analyzed': 1284.0}
```

[打开矢量 PDF 图：03b_cluster_annotation_confusion](results/figures/03b_cluster_annotation_confusion.pdf)

除 ARI/NMI 外，本流程还计算 PCA 空间中的 silhouette 分数和空间 6-近邻同类比例，用于同时评估表达可分性和空间连续性。

,author_region,names,scores,logfoldchanges,pvals,pvals_adj,gene_symbol,functional_annotation
0,Cortex_1,MtrunA17Chr5g0421871,3.005337,0.054261,2.652868e-03,7.976999e-02,NaN,NaN
1,Cortex_1,MtrunA17Chr5g0422111,1.948741,-0.081969,5.132642e-02,6.712601e-01,NaN,NaN
2,Cortex_1,MtrunA17Chr5g0422321,1.890067,-0.255186,5.874897e-02,7.396684e-01,NaN,NaN
3,Cortex_1,MtrunA17Chr4g0001901,0.483697,2.021408,6.286010e-01,9.969728e-01,NaN,NaN
4,Cortex_1,MtrunA17Chr7g0267421,0.480358,1.932238,6.309730e-01,9.969728e-01,NaN,NaN
25,Vascular bundle,MtrunA17Chr8g0360121,9.734435,2.308754,2.150087e-22,4.195895e-18,NaN,NaN
26,Vascular bundle,MtrunA17Chr7g0238571,8.315303,1.982950,9.151842e-17,8.929909e-13,NaN,NaN
27,Vascular bundle,MtrunA17Chr1g0200301,7.526471,1.272622,5.213005e-14,2.034636e-10,NaN,NaN
28,Vascular bundle,MtrunA17Chr3g0123601,7.404914,2.481844,1.312353e-13,4.268430e-10,NaN,NaN
29,Vascular bundle,MtrunA17Chr3g0145051,7.378160,1.415587,1.604920e-13,4.474288e-10,NaN,NaN


## 3. 标记基因与区域解释

标记基因来自论文补充表 Table S1-5，并使用 Mt5.0 官方注释表把基因名解析为数据中的 locus ID。
对每个区域保留一个或两个在本次 P3 样本中可检测的标记。

In [5]:
mapping = load_marker_mapping(annotation_path)
supplementary_evidence = load_supplementary_marker_evidence(supplementary_path)
marker_table = resolve_markers(
    config["marker_genes"],
    config["marker_preferred_loci"],
    mapping,
    set(plant.var_names),
    supplementary_evidence,
)
marker_table.to_csv(
    paths.table_dir / "marker_resolution.csv",
    index=False,
    encoding="utf-8-sig",
)
display(marker_table)

marker_summary = plot_marker_genes(
    plant,
    marker_table,
    config,
    paths.figure_dir / "04_marker_dotplots.pdf",
    paths.figure_dir / "05_marker_spatial.pdf",
    paths.table_dir / "marker_specificity.csv",
)
display(marker_summary)
show_pdf_figure(paths.figure_dir / "04_marker_dotplots.pdf")
show_pdf_figure(paths.figure_dir / "04_marker_dotplots_by_cluster.pdf")
show_pdf_figure(paths.figure_dir / "05_marker_spatial.pdf")

,zone,requested_acronym,resolved_gene_id,matched_acronym,description,available_in_sample,supplementary_evidence_match,supplementary_annotation
0,meristematic_zone,MtPLT3,MtrunA17Chr5g0412151,MtPLT3,plethora ERF,True,True,plethora ERF
1,meristematic_zone,MtPLT4,MtrunA17Chr7g0249621,MtPLT4|MtBBM,plethora ERF|BABYBOOM,True,True,"plethora ERF, BABYBOOM"
2,proximal_infection_zone,MtNOD25,MtrunA17Chr3g0100391,MtNOD25|MtN22,nodulin 25|Mtnodulin-22 (CAA75576),True,True,"nodulin 25, Mtnodulin-22"
3,proximal_infection_zone,MtLb6,MtrunA17Chr5g0435621,MtLb6|MtLb8,leghemoglobin6,True,True,"new nomenclature, leghemoglobin6"
4,distal_infection_zone,MtENOD8.1,MtrunA17Chr1g0160031,MtENOD8.1,early nodulin 8|AAC26810,True,True,early nodulin
5,interzone,MtSymCRK,MtrunA17Chr3g0119041,MtSymCRK,symbiotic cys-rich receptor-like kinase,True,True,symbiotic cys-rich receptor-like kinase
6,nitrogen_fixation_zone,MtNCR160,MtrunA17Chr2g0309591,MtNCR160,Nodule-specific Cysteine-Rich peptide,True,True,Nodule-specific Cysteine-Rich peptide
7,nitrogen_fixation_zone,MtLP7,MtrunA17Chr7g0236321,MtLP7,LEED..PEED family,True,True,LEED..PEED family
8,vascular_bundle,MtPHO1.1,MtrunA17Chr1g0166651,MtPHO1.1,Phosphate transporter1.1 (export),True,True,Phosphate transporter1.1


,zone,acronym,gene_id,best_author_region,mean_expression_best_region,expected_region_mean_expression,other_regions_mean_expression,expected_region_specificity_log2fc,detection_fraction_best_region,detection_fraction_overall,supplementary_evidence_match,expected_region,matches_expected_region
0,meristematic_zone,MtPLT3,MtrunA17Chr5g0412151,Meristematic zone,0.536042,0.536042,0.052561,3.350246,0.317073,0.035826,True,Meristematic zone,True
1,meristematic_zone,MtPLT4,MtrunA17Chr7g0249621,Meristematic zone,0.174738,0.174738,0.014359,3.605087,0.121951,0.013240,True,Meristematic zone,True
2,proximal_infection_zone,MtNOD25,MtrunA17Chr3g0100391,Distal infection zone,4.746180,4.457518,3.345303,0.414104,0.993671,0.917445,True,Proximal infection zone,False
3,proximal_infection_zone,MtLb6,MtrunA17Chr5g0435621,Distal infection zone,3.521213,2.424497,1.995103,0.281222,0.962025,0.666667,True,Proximal infection zone,False
4,distal_infection_zone,MtENOD8.1,MtrunA17Chr1g0160031,Distal infection zone,3.162200,3.162200,1.415816,1.159294,0.936709,0.538162,True,Distal infection zone,True
5,interzone,MtSymCRK,MtrunA17Chr3g0119041,Proximal infection zone,0.944541,0.435030,0.429277,0.019207,0.567901,0.174455,True,Interzone,False
6,nitrogen_fixation_zone,MtNCR160,MtrunA17Chr2g0309591,Distal infection zone,1.013211,0.724675,0.432028,0.746207,0.436709,0.184579,True,Nitrogen fixation zone,False
7,nitrogen_fixation_zone,MtLP7,MtrunA17Chr7g0236321,Distal infection zone,1.382502,0.572234,0.488180,0.229193,0.550633,0.192368,True,Nitrogen fixation zone,False
8,vascular_bundle,MtPHO1.1,MtrunA17Chr1g0166651,Vascular bundle,0.024863,0.024863,0.003741,2.732162,0.010811,0.006231,True,Vascular bundle,True


[打开矢量 PDF 图：04_marker_dotplots](results/figures/04_marker_dotplots.pdf)

[打开矢量 PDF 图：04_marker_dotplots_by_cluster](results/figures/04_marker_dotplots_by_cluster.pdf)

[打开矢量 PDF 图：05_marker_spatial](results/figures/05_marker_spatial.pdf)

## 4. 植物–根瘤菌信号

每个 bin 的根瘤菌占比定义为：

$$
\mathrm{rhizobia\ fraction}_i =
\frac{\mathrm{rhizobia\ counts}_i}
{\mathrm{plant\ counts}_i + \mathrm{rhizobia\ counts}_i}
$$

该指标受捕获效率、RNA 组成和组织密度影响，不能解释为细菌数量或固氮效率。

In [6]:
rhizobia_summary = plot_rhizobia_analysis(
    adata,
    paths.figure_dir / "06_rhizobia_fraction.pdf",
    paths.table_dir / "rhizobia_by_region.csv",
    paths.table_dir / "rhizobia_statistics.csv",
)
display(rhizobia_summary)
show_pdf_figure(paths.figure_dir / "06_rhizobia_fraction.pdf")
display(pd.read_csv(paths.table_dir / "rhizobia_statistics.csv"))

,organ,n_bins,median_rhizobia_fraction,q25_rhizobia_fraction,q75_rhizobia_fraction,median_total_counts,median_rhizobia_counts
0,Cortex_1,144,0.004545,0.001861,0.008922,571.5,3.0
1,Cortex_2,368,0.003223,0.001391,0.005405,2414.5,7.0
2,Distal infection zone,158,0.031564,0.017435,0.047965,2078.5,71.0
3,Interzone,59,0.046237,0.017552,0.077043,1043.0,46.0
4,Meristematic zone,41,0.001608,0.000616,0.003870,7353.0,11.0
5,Nitrogen fixation zone,63,0.006924,0.004111,0.018135,2685.0,17.0
6,Proximal infection zone,81,0.020250,0.010320,0.029336,5530.0,108.0
7,Vascular bundle,370,0.003899,0.002328,0.006637,1507.5,7.0


[打开矢量 PDF 图：06_rhizobia_fraction](results/figures/06_rhizobia_fraction.pdf)

,test,comparison,n_group_1,n_group_2,statistic,p_value,interpretation
0,Mann-Whitney U,Nitrogen fixation zone vs all other regions,63,1221,48309.500000,6.009696e-04,descriptive; spatial bins are not independent ...
1,Kruskal-Wallis,all author tissue regions,8,1284,554.483163,1.531089e-115,descriptive; spatial bins are not independent ...


In [7]:
adata.obs.to_csv(
    paths.table_dir / "bin_level_results.csv",
    encoding="utf-8-sig",
)
write_run_metadata(
    adata,
    plant,
    metrics,
    config,
    sample_reference,
    paths.table_dir / "run_metadata.json",
)

## 结论边界

- 本复现证明公开矩阵可以被独立读取、重建 QC、重新聚类、映射标记并拆分物种计数。
- 独立聚类不是论文注释的复制品。分辨率 0.8 的空间结构清晰，但 ARI 只有约 0.20，
  说明作者注释整合了本次单切片之外的证据；不能通过挑选参数追求图形完全一致。
- 根瘤菌占比在感染区/交界区较高，在分生区较低。该结果支持“宿主组织分区与细菌信号相关”，
  但不能推断因果关系。
- 每个 bin 不是独立生物学重复，因此区域统计只作描述性辅助。
- 补充表 S1-3 报告 1,270 个 cell/bin，公开矩阵为 1,284 个；本分析保留公开对象
  的全部 bin，并在元数据中记录该上游差异。
- 论文完整的 20 个切片、Harmony 整合、RNA velocity、Monocle 3、WGCNA 和实验验证不在此范围内。